# Membre 2 - Streaming & Mobilité CityBikes

Ce notebook présente et lance le pipeline de collecte mobilité pour **Paris, Lyon et Lille**. Le code métier est partagé avec le script `scripts/citybikes_stream.py`, ce qui permet de lancer le même pipeline en continu hors de Jupyter.

Le pipeline produit :

1. **Bronze** : payloads JSON bruts partitionnés par réseau et date ;
2. **Silver** : stations typées en Parquet, avec timestamp UTC et déduplication ;
3. **Gold mobilité** : dernier état par station et indicateurs horaires.

## 1. Paramétrage

La fréquence de collecte retenue est de 5 minutes, soit 300 secondes. Cette cadence limite aussi le risque de rate limiting CityBikes.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'cities.yml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.citybikes_stream import (
    TARGET_CITIES,
    build_session,
    collect_once,
    build_minio_client,
    ensure_bucket,
    load_cities,
    read_silver,
)

CONFIG_PATH = PROJECT_ROOT / 'config' / 'cities.yml'
DATA_ROOT = PROJECT_ROOT / 'data'
cities = load_cities(CONFIG_PATH)
TIMEOUT = (5, 15)
REQUEST_DELAY_SECONDS = 1.0
STORAGE_MODE = 'minio'  # utiliser 'local' pour tester sans envoyer vers le cloud
minio_client = None
minio_buckets = None
if STORAGE_MODE != 'local':
    minio_client, minio_buckets = build_minio_client()
    for bucket in minio_buckets.values():
        ensure_bucket(minio_client, bucket)

print('Villes :', [cities[city_id]['name'] for city_id in TARGET_CITIES])
print('Réseaux :', {city_id: cities[city_id]['citybikes_network_id'] for city_id in TARGET_CITIES})
print('Sortie :', DATA_ROOT)

## 2. Collecte d'un micro-batch

Cette cellule lance un seul cycle. Pour le streaming récurrent, utiliser le script depuis un terminal afin de ne pas bloquer le notebook. Une erreur sur une ville est journalisée et n'empêche pas les autres villes d'être traitées.

In [ ]:
session = build_session()
collected_rows = collect_once(
    session=session,
    cities=cities,
    output_root=DATA_ROOT,
    timeout=TIMEOUT,
    request_delay=REQUEST_DELAY_SECONDS,
    minio_client=minio_client,
    minio_buckets=minio_buckets,
)
print(f'Lignes collectées pendant ce cycle : {collected_rows:,}')

## 3. Vérification de la Silver

La clé logique de déduplication est `(network_id, station_id, snapshot_minute_utc)`. Elle évite de compter deux fois une relance dans la même minute tout en conservant les observations successives.

In [ ]:
silver = read_silver(DATA_ROOT)
if silver.empty:
    print('Aucune donnée Silver disponible. Vérifier la connectivité ou le rate limiting CityBikes.')
else:
    print(f'{len(silver):,} lignes Silver')
    display(silver.groupby(['city_id', 'city_name'], as_index=False).agg(
        stations=('station_id', 'nunique'),
        observations=('station_id', 'count'),
        dernier_releve=('observed_at_utc', 'max'),
    ))
    display(silver.head())

## 4. Indicateurs Gold mobilité

Les KPI sont calculés à partir de la Silver dédupliquée et écrits dans les deux tables attendues.

Définitions :

- `mobility_station_latest.parquet` : état le plus récent de chaque station ;
- `mobility_hourly.parquet` : stocks moyens/totaux, taux de rupture, taux de saturation et stations critiques ;
- les taux sont calculés sur les observations réellement collectées.

La clé Silver de déduplication est `(network_id, station_id, snapshot_minute_utc)`.

In [ ]:
gold_dir = DATA_ROOT / 'gold' / 'kpis'
latest_path = gold_dir / 'mobility_station_latest.parquet'
hourly_path = gold_dir / 'mobility_hourly.parquet'
if latest_path.exists() and hourly_path.exists():
    mobility_latest = pd.read_parquet(latest_path)
    mobility_hourly = pd.read_parquet(hourly_path)
    print(f'mobility_station_latest : {len(mobility_latest):,} lignes')
    print(f'mobility_hourly        : {len(mobility_hourly):,} lignes')
    display(mobility_hourly.tail(10))
else:
    print('Les tables Gold seront disponibles après une collecte réussie.')

## 5. Lancement streaming hors notebook

Depuis la racine du projet :

```bash
.venv/bin/python scripts/citybikes_stream.py
```

Le cycle est exécuté toutes les 300 secondes, soit 5 minutes.

```bash
.venv/bin/python scripts/citybikes_stream.py --interval-seconds 300
```

Arrêt propre avec `Ctrl+C`. Les fichiers Bronze, Silver et Gold déjà écrits sont conservés.